# Vegetation Biophysical Products: NDVI, LAI & FAPAR

This notebook demonstrates searching and visualizing vegetation biophysical products from multiple archives:

- **CDSE** — Sentinel-3 SYN VGT-like 10-day synthesis (300 m NDVI)
- **Terrascope** — Sentinel-2-derived NDVI, LAI, and FAPAR (10–20 m)

We use the `rs_tools` package for uniform search, catalog lookup, and visualization.

In [ ]:
%matplotlib widget

import warnings
warnings.filterwarnings("ignore")

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import get_archive, search_archive
from rs_tools.datasets.catalog import get, list_datasets
from rs_tools.visualization.timeseries import plot_timeseries_slider, plot_timeseries_line
from rs_tools.visualization.globe import add_globe_inset
from rs_tools.visualization.slider import slider_comparison
from rs_tools.visualization.rgb_composite import make_rgb, multi_temporal_rgb, plot_rgb

import matplotlib.pyplot as plt
import numpy as np

## 1. Browse the catalog

List all biophysical products registered in the catalog.

In [ ]:
for ds in list_datasets(tag="biophysical"):
    archives = ", ".join(ds.archive_collections.keys())
    print(f"{ds.short_name:20s}  {ds.spatial_resolution or '':>8s}  archives: {archives}")

## 2. Define area of interest

We use a bounding box over Belgium for this demonstration.

In [ ]:
bbox = BoundingBox(west=3.0, south=50.0, east=6.0, north=51.5)
print(f"AOI: {bbox}")

## 3. Search CDSE — Sentinel-3 300 m NDVI (SYN V10)

The CGLOPS 300 m NDVI V3 product line uses Sentinel-3 SYN (OLCI + SLSTR) data.
On CDSE, the corresponding collection is `sentinel-3-syn-2-v10-ntc` (10-day synthesis).

In [ ]:
ndvi_info = get("CGLOPS_NDVI_V3")
print(f"Product : {ndvi_info.name}")
print(f"CDSE IDs: {ndvi_info.archive_collections.get('cdse', [])}")

config_cdse = SearchConfig(
    start_date="2024-01-01",
    end_date="2024-06-30",
    bbox=bbox,
    collections=ndvi_info.archive_collections["cdse"],
    limit=20,
)

items_cdse = search_archive("cdse", config_cdse)
print(f"\nFound {len(items_cdse)} CDSE items")
for item in items_cdse[:5]:
    print(f"  {item['id']}  {item['properties'].get('datetime', '')}")

## 4. Search Terrascope — Sentinel-2 NDVI, LAI, FAPAR (10–20 m)

Terrascope hosts Sentinel-2-derived biophysical products at 10–20 m resolution.

In [ ]:
products = ["CGLOPS_NDVI_V3", "CGLOPS_LAI_V3", "CGLOPS_FAPAR_V3"]

terrascope_results = {}
for prod_name in products:
    info = get(prod_name)
    tc_collections = info.archive_collections.get("terrascope", [])
    if not tc_collections:
        continue
    config_tc = SearchConfig(
        start_date="2024-06-01",
        end_date="2024-06-30",
        bbox=bbox,
        collections=tc_collections,
        limit=10,
    )
    items = search_archive("terrascope", config_tc)
    terrascope_results[prod_name] = items
    print(f"{info.short_name}: {len(items)} items from {tc_collections}")
    for it in items[:3]:
        print(f"  {it['id']}")

## 5. Inspect item assets

Each STAC item carries asset links (COG files, thumbnails, metadata).

In [ ]:
# Show assets for the first CDSE NDVI item
if items_cdse:
    first = items_cdse[0]
    print(f"Item: {first['id']}")
    print(f"Datetime: {first['properties'].get('datetime')}")
    print(f"Assets:")
    for name, asset in first.get("assets", {}).items():
        print(f"  {name:30s}  {asset.get('type', 'n/a'):30s}  {asset.get('href', '')[:80]}")

In [ ]:
# Show assets for first Terrascope NDVI item
if terrascope_results.get("CGLOPS_NDVI_V3"):
    first = terrascope_results["CGLOPS_NDVI_V3"][0]
    print(f"Item: {first['id']}")
    print(f"Assets:")
    for name, asset in first.get("assets", {}).items():
        print(f"  {name:30s}  {asset.get('type', 'n/a'):30s}  {asset.get('href', '')[:80]}")

## 6. Visualize temporal coverage

Plot a simple timeline showing when data is available.

In [ ]:
from datetime import datetime as dt

fig, ax = plt.subplots(figsize=(12, 3))

y_pos = 0
labels = []

# CDSE NDVI dates
if items_cdse:
    dates_cdse = []
    for it in items_cdse:
        d = it["properties"].get("datetime") or it["properties"].get("start_datetime", "")
        if d:
            dates_cdse.append(dt.fromisoformat(d.replace("Z", "+00:00")))
    if dates_cdse:
        ax.scatter(dates_cdse, [y_pos] * len(dates_cdse), marker="|", s=200, label="CDSE NDVI 300m")
        y_pos += 1

# Terrascope products
for prod_name, items in terrascope_results.items():
    dates = []
    for it in items:
        d = it["properties"].get("datetime") or it["properties"].get("start_datetime", "")
        if d:
            dates.append(dt.fromisoformat(d.replace("Z", "+00:00")))
    if dates:
        short = prod_name.replace("CGLOPS_", "TC ")
        ax.scatter(dates, [y_pos] * len(dates), marker="|", s=200, label=short)
        y_pos += 1

ax.set_yticks(range(y_pos))
ax.set_yticklabels([])
ax.legend(loc="upper left")
ax.set_title("Temporal coverage — Belgium AOI")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7. Location context — 3-D globe inset

Add a globe inset to confirm the area of interest.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(bbox.west - 1, bbox.east + 1)
ax.set_ylim(bbox.south - 1, bbox.north + 1)
ax.set_title("Area of Interest — Belgium")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Draw the AOI rectangle
from matplotlib.patches import Rectangle
rect = Rectangle(
    (bbox.west, bbox.south),
    bbox.east - bbox.west,
    bbox.north - bbox.south,
    linewidth=2, edgecolor="red", facecolor="red", alpha=0.2,
)
ax.add_patch(rect)

# Add globe inset
add_globe_inset(fig, bbox)
plt.show()

## Next steps

- **Download data** using the asset HREFs (COG/NetCDF) and load with `rioxarray` or `xarray`
- **Time-series slider** (`plot_timeseries_slider`) for stepping through NDVI maps over time
- **Slider comparison** between NDVI and LAI or FAPAR for the same date
- **Multi-temporal RGB** composite from three NDVI dates